# Ngày 29 · Dự án tổng kết: khóa holdout và baseline

- Tách và khóa một tập holdout trước khi nhìn dữ liệu, ghi lại dấu vân tay của nó
- Đo bằng đúng thước đo của cuộc thi (House Prices: RMSE trên log giá)
- Dựng thang baseline: dự đoán trung bình → tuyến tính một biến → tuyến tính đầy đủ → mô hình mạnh

**Đọc:**
- Ng (2018), Machine Learning Yearning, các chương về tập dev/test và chọn thước đo một con số
- Kaufman, Rosset, Perlich & Stitelman (2012), Leakage in Data Mining: Formulation, Detection, and Avoidance, ACM TKDD 6(4)
- Dwork et al. (2015), The Reusable Holdout: Preserving Validity in Adaptive Data Analysis, Science 349(6248), 636–638
- Kaggle, House Prices: Advanced Regression Techniques, mục Evaluation

In [ ]:
import hashlib, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if (p / 'real_estate.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Mở notebook từ thư mục py30 hoặc py30/bai_lam để thấy data/real_estate.csv.')
# Real estate valuation (Yeh, 2018; UCI, CC BY 4.0): 414 giao dịch, giá theo 10.000 TWD/ping.
real_estate = pd.read_csv(DATA / 'real_estate.csv')
TARGET = 'price_10k_ntd_per_ping'

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

def _cv_rmse_ref(make_model, X, y, seed):
    out = []
    for tr, va in KFold(5, shuffle=True, random_state=seed).split(X):
        m = make_model().fit(X.iloc[tr], y[tr]); out.append(np.sqrt(np.mean((m.predict(X.iloc[va]) - y[va]) ** 2)))
    return float(np.mean(out))
from sklearn.metrics import mean_squared_log_error
def _lock_ref(n, frac, seed):
    o = np.random.default_rng(seed).permutation(n); k = int(round(frac * n)); return np.sort(o[k:]), np.sort(o[:k])


## Khóa holdout và dấu vân tay

Gợi ý: np.random.default_rng(seed).permutation(n); k = int(round(frac·n)); holdout = k chỉ số đầu, dev = phần còn lại, cả hai sắp xếp tăng dần. fingerprint: hashlib.sha256 trên np.asarray(sorted(idx), dtype=np.int64).tobytes(), lấy 16 ký tự hex đầu.

In [ ]:
def lock_holdout(n, frac, seed):
    # Trả (dev_idx, holdout_idx), mỗi mảng đã sắp xếp tăng dần.
    raise NotImplementedError

def fingerprint(idx):
    # Chuỗi 16 ký tự hex, không phụ thuộc thứ tự phần tử trong idx.
    raise NotImplementedError


In [ ]:
_report('d29-holdout',
    _check('Chia đúng cỡ, rời nhau, phủ đủ', lambda: ((lambda d, h: len(h) == 83 and len(d) == 331 and np.array_equal(np.sort(np.r_[d, h]), np.arange(414)) and np.all(np.diff(h) > 0))(*lock_holdout(414, 0.2, 0)), '414 dòng, 20% holdout.')),
    _check('Lặp lại được theo seed và có xáo trộn', lambda: (all(np.array_equal(a, b) for a, b in zip(lock_holdout(414, .2, 0), _lock_ref(414, .2, 0))) and not np.array_equal(lock_holdout(414, .2, 0)[1], lock_holdout(414, .2, 1)[1]), 'Khớp phép chia tham chiếu với seed 0.')),
    _check('Dấu vân tay ổn định và nhạy với thay đổi', lambda: ((lambda h: fingerprint(h) == '167ee3b5c648a47e' and fingerprint(h[::-1]) == fingerprint(h) and fingerprint(np.r_[h[1:], 0]) != fingerprint(h))(lock_holdout(414, .2, 0)[1]), 'Holdout seed 0 có dấu vân tay 167ee3b5c648a47e.')),
)


## Thước đo cuộc thi và RMSE cross-validation

Gợi ý: rmsle = √mean((log1p(p) − log1p(y))²), chặn p ≥ 0 trước khi lấy log. cv_rmse: với mỗi fold của KFold(5, shuffle=True, random_state=seed), gọi make_model() để có mô hình MỚI, fit trên fold train, tính RMSE trên fold validation, rồi lấy trung bình 5 fold.

In [ ]:
def rmse(y, p):
    raise NotImplementedError

def rmsle(y, p):
    # Thước đo của House Prices: RMSE giữa log(1 + dự đoán) và log(1 + giá thật).
    raise NotImplementedError

def cv_rmse(make_model, X, y, seed):
    # X: DataFrame, y: mảng. make_model() trả mô hình chưa fit.
    raise NotImplementedError


In [ ]:
# Tập phát triển cố định (80%) — 20% còn lại là holdout đã khóa ở Ngày 29.
_order = np.random.default_rng(0).permutation(len(real_estate))
_hold = np.sort(_order[:83]); _dev = np.sort(_order[83:])
dev = real_estate.iloc[_dev].reset_index(drop=True)
X_dev, y_dev = dev.drop(columns=TARGET), dev[TARGET].to_numpy()
_report('d29-baseline',
    _check('rmsle khớp scikit-learn và chịu được dự đoán âm', lambda: (np.isclose(rmsle([200000, 150000, 310000], [210000, 140000, 300000]), np.sqrt(mean_squared_log_error([200000, 150000, 310000], [210000, 140000, 300000]))) and np.isfinite(rmsle([10.], [-5.])), 'Chặn dự đoán ở 0.')),
    _check('cv_rmse khớp cross_val_score (neg RMSE)', lambda: (np.isclose(cv_rmse(LinearRegression, X_dev, y_dev, 0), -cross_val_score(LinearRegression(), X_dev, y_dev, cv=KFold(5, shuffle=True, random_state=0), scoring='neg_root_mean_squared_error').mean()), 'Tuyến tính 6 biến trên tập dev.')),
    _check('Baseline trung bình ≈ độ lệch chuẩn của y', lambda: ((lambda r: (abs(r - y_dev.std()) < .5, f'RMSE {r:.2f}, std {y_dev.std():.2f}.'))(cv_rmse(DummyRegressor, X_dev, y_dev, 0))[0], 'DummyRegressor dự đoán trung bình của fold train.')),
)


## Dự án

Chọn cuộc thi Kaggle cho dự án tổng kết (mặc định House Prices; không cần liên quan luận văn). Mở notebook py30/cloud/Ngay_29_30_Capstone (Kaggle hoặc Colab). Khóa 20% train làm holdout bằng lock_holdout, ghi fingerprint vào README dự án và hứa không đánh giá trên đó cho tới cuối Ngày 30. Báo cáo thang baseline bằng RMSLE 5-fold trên phần dev, rồi nộp submission của baseline tốt nhất để có mốc điểm Kaggle.